# Decode and rename data

`data/processed/dua.parquet` stores one row per declaration item, with every column still named and coded exactly as DNA publishes it (`DD*` tags from the XML spec). This step does three things, all written to a brand-new file (`dua_decoded.parquet`) so the source parquet from 0.1 is never touched. The code (`decode_dataset`, with the catalogs and new names below as `DECODED_COLUMNS`, `TIPO_DOCUM`, `REGIMENES`, `SUBREGIMENES` and `RENAME_COLUMNS`) lives in `uruguay_dua_comex/pipeline.py`, shared with the automatic update:

1. **Decode in place**: `TIPO_REGIMEN` (I/E/T) is overwritten with its description, since the query filters on it.
2. **Rename**: give every column an intuitive name instead of DNA's internal tag. Two naming choices (`TIPO_REGIMEN`/`REGIMEN_ADUANERO` and `PAIS_PROCEDENCIA_DESTINO`) had several reasonable options and were picked by the project owner (see notes in the table below). `DDUNID_TRANS` is left with its raw name, since its meaning isn't documented anywhere found.
3. **Name codes**: seven coded columns keep DNA's code and get a `<column>_NOMBRE` column right after it with the official description, verbatim from DNA (only extra spaces removed). A code with no official match gets a null name, never a guessed one; the code itself is always kept.

## Official sources

DNA publishes the code tables for several DUA fields on its public FTP, and listed them, with the codes for the remaining fields, in an annex to an access-to-information request answered by the Ministerio de Economía y Finanzas (expediente 2025-5-1-0004266, resolution of 2025-06-26; DNA's annex, Área Tecnologías de la Información, dated 2025-06-23). Only what those sources confirm is named; every other coded column is left as DNA's raw code.

- **Countries and free zones** (`PAIS_ORIGEN`, `PAIS_PROCEDENCIA_DESTINO`): `ftp://ftp.aduanas.gub.uy/Datos%20Basicos/rgpais.zip`. A renamed code has one row per name, each with its start date; the name with the latest start date is used ("current name", project owner's choice), even for codes no longer in use (e.g. `891` SERBIA Y MONTENEGRO).
- **Customs offices** (`ADUANA_INGRESO_EGRESO`): `ftp://ftp.aduanas.gub.uy/Datos%20Basicos/rgaduan.zip` (an Excel file despite its `.xml` name).
- **Transport modes** (`VIA_TRANSPORTE`): `ftp://ftp.aduanas.gub.uy/Datos%20Basicos/dbviatr.zip`.
- **Document type, regime and subregime** (`TIPO_DOCUMENTO_IMPOEXPO`, `REGIMEN_ADUANERO`, `SUBREGIMEN`): transcribed from DNA's annex, saved as `docs/codificacion-campos-dua-2025.pdf` (source: https://www.aduanas.gub.uy/innovaportal/file/27289/1/expediente-2025-5-1-0004266-diego-tenenbaum-lion.pdf, retrieved 2026-11-05). A subregime code is looked up together with its regime (`10-21` and `20-21` are listed separately).
- Field descriptions for the rename column: `docs/FormatoDUADiariosPublicos.htm` (DNA's own DUA XML spec).

The FTP tables are downloaded by `0.0` (and by the automatic update before every rebuild) into `data/raw/catalogos/`, and `decode_dataset` uses the newest local copy, downloading them first if they're missing.

Codes with no official match, as of the 2026-11 rebuild: `PAIS_ORIGEN` `000` (350 import rows, 2016–2018) and subregimes `10-TT`, `10-1U`, `10-14`, `40-44` and `40-45` (1,187 rows). The log of `decode_dataset` reports these counts on every run.

Earlier versions of this notebook relied on the import/export DUA instructivos for three partial catalogs (document type 2/4/6, offices 001/002/009, transport mode A); the annex and FTP tables supersede them and agree with them on every code they shared.

| Raw column | New name | Decoded? | Notes |
|---|---|---|---|
| `DDANO_PRESE` | `ANIO_PRESENTACION` | No | Year the DUA was filed |
| `DDNUME_CORRE_PUBLICO` | `NUMERO_PUBLICO_DUA` | No | Repeats every year; combine with `ANIO_PRESENTACION` to identify a declaration (see 0.1) |
| `DDFECH_INGSI` | `FECHA_DUA` | No | |
| `DDTIPO_REGI` | `TIPO_REGIMEN` | Yes (I/E/T → Importación/Exportación/Tránsito) | Kept distinct from `REGIMEN_ADUANERO` (`DDCODI_REGI`) — different concepts, both called "régimen" in Spanish; project owner's choice |
| `DDPAIS_ORIGE` | `PAIS_ORIGEN` | Named (`PAIS_ORIGEN_NOMBRE`, from rgpais) | |
| `DDPART_NANDI` | `NCM` | No — 10-digit tariff code, too large to embed. Official lookup: [NCM classification](https://www.aduanas.gub.uy/innovaportal/v/8495/3/innova.front/ncm-.html), [Arancel Nacional](https://www.gub.uy/ministerio-economia-finanzas/politicas-y-gestion/nomenclatura-aranceles-uruguay) (MEF) | |
| `DDPUER_EMBAR` | `PAIS_PROCEDENCIA_DESTINO` | Named (`PAIS_PROCEDENCIA_DESTINO_NOMBRE`, from rgpais) | País de procedencia for imports, país destino for exports — one field, dual meaning; project owner's choice. Includes Uruguayan free zones (`980`–`997`) as destinations |
| `DDCONV_INTER` | `CONVENIO_INTERNACIONAL` | No — DNA confirms an "Acuerdos" table exists; only per-agreement *document* codes were found (Anexo II above), not the agreement code itself | |
| `DDCODI_LIBER` | `CODIGO_EXONERACION` | No — table exists, no published values found | |
| `DDQUNICOM` | `CANTIDAD_UNIDADES_COMERCIALES` | No | |
| `DDTUNICOM` | `TIPO_UNIDAD_COMERCIAL` | No — table exists, no published values found | |
| `DDUNID_FIQTY` | `CANTIDAD_UNIDADES_FISICAS` | No | |
| `DDUNID_FIDES` | `TIPO_UNIDAD_FISICA` | No — defined per-NCM-item in the Arancel Nacional itself, no standalone catalog | |
| `DDVAD_INCR` | `VALOR_ADUANA` | No | |
| `DDIMA_DOLAR` | `IMADUNI_USD` | No | Acronym kept as-is; no verified source for its full expansion |
| `DDLIMA_DOLAR` | `IMADUNI_LIBERADO_USD` | No | |
| `DDRMI_DOLAR` | `RECARGO_MINIMO_USD` | No | |
| `DDLRMI_DOLAR` | `RECARGO_MINIMO_LIBERADO_USD` | No | |
| `DDRAD_DOLAR` | `RECARGO_ADICIONAL_USD` | No | |
| `DDLRAD_DOLAR` | `RECARGO_ADICIONAL_LIBERADO_USD` | No | |
| `DDRMO_DOLAR` | `RECARGO_MOVIL_USD` | No | |
| `DDLRMO_DOLAR` | `RECARGO_MOVIL_LIBERADO_USD` | No | |
| `DDIVA_DOLAR` | `IVA_USD` | No | |
| `DDLIVA_DOLAR` | `IVA_LIBERADO_USD` | No | |
| `DDLIVAA_DOLA` | `IVA_ANTICIPO_LIBERADO_USD` | No | |
| `DDIVAA_DOLAR` | `IVA_ANTICIPO_USD` | No | |
| `DDPOR_IMADUN` | `IMADUNI_PORCENTAJE` | No | |
| `DDPOR_RMI` | `RECARGO_MINIMO_PORCENTAJE` | No | |
| `DDPOR_RAD` | `RECARGO_ADICIONAL_PORCENTAJE` | No | |
| `DDPOR_IVA` | `IVA_PORCENTAJE` | No | |
| `DDPOR_IVAA` | `IVA_ANTICIPO_PORCENTAJE` | No | |
| `DDVIA_TRANSP` | `VIA_TRANSPORTE` | Named (`VIA_TRANSPORTE_NOMBRE`, from dbviatr) | |
| `DDUNID_TRANS` | *(unchanged)* | No | Not in DNA's official field spec at all (only published 2016–2019); meaning unverified anywhere found, so left as the raw DNA tag rather than guessed at |
| `DDADUAINGEGR` | `ADUANA_INGRESO_EGRESO` | Named (`ADUANA_INGRESO_EGRESO_NOMBRE`, from rgaduan) | |
| `DDTIPO_DOCUM` | `TIPO_DOCUMENTO_IMPOEXPO` | Named (`TIPO_DOCUMENTO_IMPOEXPO_NOMBRE`, from the annex) | |
| `DDLIBR_TRIBU` | `NUMERO_DOCUMENTO_IMPOEXPO` | No | The importer/exporter's actual document number (RUT, CI, etc.), paired with `TIPO_DOCUMENTO_IMPOEXPO` |
| `DDPESO_BRUTO` | `PESO_BRUTO` | No | |
| `DDPESO_NETO` | `PESO_NETO` | No | |
| `DDDNOMBRE` | `NOMBRE_IMPORTADOR_EXPORTADOR` | No | |
| `DDCODI_REGI` | `REGIMEN_ADUANERO` | Named (`REGIMEN_ADUANERO_NOMBRE`, from the annex) | Kept distinct from `TIPO_REGIMEN` (`DDTIPO_REGI`); project owner's choice. Published since 2020 |
| `DDTIPO_OPERA` | `SUBREGIMEN` | Named (`SUBREGIMEN_NOMBRE`, from the annex, keyed by regime) | Published since 2020 |
| `FUENTE_PERIODO`, `FUENTE_TIPO` | *(unchanged)* | — | Added by 0.1's own pipeline, not a DNA field — already intuitive |

For the codes still left raw (agreements, exemptions, units), DNA's "Consultas DUA" tool (https://www.aduanas.gub.uy/innovaportal/v/18714/6/innova.front/consultas-dua.html) lets you filter by the same fields, which is a way to check a code by eye.

Numeric columns are already typed from 0.1 (`Float64`/`Date`/`Int16`), so no casting happens here.

In [ ]:
import logging

import polars as pl

from uruguay_dua_comex import utils
from uruguay_dua_comex.pipeline import NAME_SUFFIX, decode_dataset
from uruguay_dua_comex.query import DECODED_FILENAME

logging.basicConfig(level=logging.INFO, format="%(message)s")

# Logs which catalog files are used and how many rows have a code with no
# official name, then writes data/processed/dua_decoded.parquet.
decode_dataset()

# Preview each named code next to its name, from the written output.
decoded = pl.scan_parquet(utils.data_processed_dir(DECODED_FILENAME))
name_columns = [c for c in decoded.collect_schema().names() if c.endswith(NAME_SUFFIX)]
preview_columns = ["TIPO_REGIMEN"]
for name_column in name_columns:
    preview_columns += [name_column.removesuffix(NAME_SUFFIX), name_column]
decoded.select(preview_columns).head().collect()